# F01 â€” Build and execute CUDA in a T4 notebook

Run this notebook in Colab with a T4 runtime or Kaggle with a T4 accelerator and internet access. All compilation and GPU execution happen in this remote Linux session. Your local computer only needs a browser/editor and Git.

This notebook builds a native **vector smoke operation**, not native RMSNorm. It checks `y = 3 Ã— x`, then runs the PyTorch RMSNorm reference on CUDA tensors. No performance claim follows from a smoke pass. F02 will implement CUDA RMSNorm.

Each code cell explains its purpose. Run in order. If setup or a test fails, run the final export cell anyway and return the archive. A hard session loss may leave incomplete artifacts; it is never a pass.

## 1. Pin the source

A commit SHA identifies the exact source being tested. Copy the full 40-character commit from the feature PR, rather than testing a moving branch. A fresh directory prevents overwriting a previous attempt. This public checkout needs no GitHub token.

In [ ]:
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import tempfile
import venv

REVISION = "PASTE_FULL_COMMIT_SHA"
assert re.fullmatch(r"[0-9a-fA-F]{40}", REVISION), "Set REVISION to the PR's full commit SHA"
base = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
assert base.is_dir(), "Use a Colab or Kaggle Linux notebook"
workspace = Path(tempfile.mkdtemp(prefix="aegis-f01-", dir=base))
repo = workspace / "repo"
artifacts = workspace / "artifacts"
artifacts.mkdir()

def run(args, *, cwd=None, env=None):
    completed = subprocess.run(args, cwd=cwd, env=env or globals().get("project_env"), text=True, capture_output=True)
    with (artifacts / "setup-and-tests.log").open("a") as log:
        log.write("\n$ " + " ".join(map(str, args)) + "\n")
        log.write(completed.stdout + completed.stderr)
    print((completed.stdout + completed.stderr)[-4000:])
    completed.check_returncode()
    return completed

run(["git", "init", str(repo)])
run(["git", "remote", "add", "origin", "https://github.com/MutugiD/Aegis-Norm.git"], cwd=repo)
run(["git", "fetch", "--depth=1", "origin", REVISION], cwd=repo)
run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
actual = run(["git", "rev-parse", "HEAD"], cwd=repo).stdout.strip()
assert actual.lower() == REVISION.lower()
(artifacts / "commit.txt").write_text(actual + "\n")
print("Workspace:", workspace)

## 2. Inspect the notebook host

The **driver** lets the OS communicate with the GPU. The **CUDA toolkit** supplies `nvcc`, which compiles CUDA source on the VM's CPU. The **PyTorch wheel** supplies its own CUDA runtime dependencies; installing it does not install a complete compiler toolchain.

C2 uses PyTorch 2.14.0/cu126 and toolkit 12.6. This remains a qualification candidate. This cell stops before large downloads if the toolkit is missing or different. Do not force a driver/toolkit replacement just to bypass the check; export the report for compatibility review.

In [ ]:
run(["nvidia-smi"])
nvcc = shutil.which("nvcc") or "/usr/local/cuda/bin/nvcc"
toolkit = run([nvcc, "--version"]).stdout
assert re.search(r"release 12\.6\b", toolkit), "C2 expects toolkit 12.6; export the actual environment"
run(["c++", "--version"])

## 3. Install in an isolated notebook environment

A Python virtual environment keeps project dependencies separate from provider-installed packages. Its CPU compiler and GPU are still those of the notebook VM. We explicitly choose the cu126 wheel, then install the package and its build/test dependencies. The package wheel includes native source files; the next step compiles them for this session.

This can take several minutes and requires disk space for PyTorch and its dependencies. Model weights are not downloaded in F01.

In [ ]:
environment = workspace / "venv"
venv.EnvBuilder(with_pip=True).create(environment)
python = str(environment / "bin" / "python")
project_env = os.environ.copy()
project_env["PATH"] = str(environment / "bin") + os.pathsep + project_env["PATH"]
run([python, "-m", "pip", "install", "torch==2.14.0",
     "--index-url", "https://download.pytorch.org/whl/cu126"])
run([python, "-m", "pip", "install", "-r", str(repo / "requirements-foundation.txt")])
run([python, "-m", "pip", "install", "--no-deps", str(repo)])
run([python, "-m", "pip", "check"])
run([python, "-m", "pip_audit", "--format", "json", "--output", str(artifacts / "installed-audit.json")])
run([python, "-m", "pip_audit", "-r", str(repo / "requirements-foundation.txt"),
     "--no-deps", "--disable-pip", "--format", "json", "--output", str(artifacts / "direct-audit.json")])

## 4. Preflight and explicit native build

Preflight records the actual GPU, available memory, Python/PyTorch versions, compiler, toolkit and driver. `ready_for_build` means the build may be attempted; it is not a correctness result.

The smoke command invokes the C++/CUDA extension builder. Ninja coordinates compilation, `MAX_JOBS=2` limits host build parallelism, and architecture `7.5` targets T4. The binding receives a PyTorch tensor and passes its existing GPU pointer to the kernel; it does not copy the tensor through CPU memory.

The CUDA launcher uses PyTorch's current stream. Returning to Python does not mean the GPU is finished. The checks synchronize before comparing values, and exercise a non-default stream. Compiler output is saved to `build.log`; this cell may be quiet while compilation runs.

In [ ]:
run([python, "-m", "aegis_norm.preflight", "--output",
     str(artifacts / "preflight.json"), "--require-t4"], cwd=repo)
run([python, "-m", "aegis_norm.smoke", "--output-root", str(artifacts)], cwd=repo)

## 5. Run reference and native binding tests

The **reference** calculates RMSNorm with ordinary PyTorch operations. When its input tensor is on CUDA, those operations run on the T4. It accumulates in FP32 and casts normalized values before multiplying by the weight, preserving the specified rounding boundary.

`backend='cuda'` intentionally fails for RMSNorm in F01: the vector smoke kernel must never masquerade as native RMSNorm. `explain_dispatch` exposes that distinction. GPU tests require explicit opt-in so a CPU CI job cannot silently claim to have qualified CUDA.

In [ ]:
test_env = project_env.copy()
test_env["AEGIS_RUN_GPU"] = "1"
run([python, "-m", "pytest", str(repo / "tests/test_reference.py"),
     str(repo / "tests/test_preflight.py"), str(repo / "tests/test_gpu_smoke.py"),
     "-v", "--junitxml=" + str(artifacts / "tests.xml")], cwd=workspace, env=test_env)

## 6. Export evidence, including failures

Run this cell even if an earlier cell failed. The archive contains the commit, setup/test log, environment report and any smoke results. A successful smoke run additionally contains compiler output, case results, resolved package versions and SHA-256 file hashes.

Resolved versions are an environment snapshot, not proof that a different notebook will reproduce the run. Maintain the selected wheel index and native toolchain as recorded. F01 qualification remains pending until the run is reviewed. Full model/benchmark recovery workflows follow in F06.

In [ ]:
archive = shutil.make_archive(str(workspace / "aegis-f01-evidence"), "zip", artifacts)
print("Evidence archive:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(archive))
    print("On Kaggle, also download the archive from the notebook output files.")